In [ ]:
%load_ext autoreload
%autoreload 2

from setup_imports import *  # noqa: F401,F403

# Tag phrases from text, then sync tags to Anki

Walks through the real end-to-end workflow with one concrete example:

1. `add_tags_from_text` — from a piece of target-language text, find the minimum set of *existing* phrases whose translation already covers its vocab, and tag those phrases in Firestore (tag stored **unprefixed**).
2. Confirm the tag landed on the phrase's translation in Firestore.
3. `sync_tag_to_anki` — sync that tag into the live Anki collection (`dry_run=True` first). It should show up on the note as `fs::food_and_drink` — the `fs::` prefix is added only at this step; Firestore itself keeps the bare `food_and_drink`.
4. Only once you're happy with the dry-run report: flip `RUN_FOR_REAL` to `True` and re-run the last cell to actually write to Anki.

In [ ]:
from phrases.search import add_tags_from_text, find_phrases_by_tag
from connections.anki_collection import get_anki_collection, close_anki_collection
from anki_sync import sync_tag_to_anki
from phrases.generation import generate_phrases_from_vocab_dict
from phrases.phrase_model import Phrase
from langcodes import Language
from anki_tools import create_anki_deck

## Config

In [ ]:
#load some text

with open("../data/text_to_process/wildfires.txt", "r", encoding="utf-8") as f:
    text = f.readlines()

text = " ".join([line.strip() for line in text if line.strip()])

In [ ]:
text

In [ ]:
TEXT = text # Swedish for "a bottle of wine"
# create an anki deck and import.
TARGET_LANGUAGE = Language.get("sv-SE")
print(TARGET_LANGUAGE.display_name())
SOURCE_LANGUAGE = Language.get("en-GB")
print(SOURCE_LANGUAGE.display_name())
TAG = "8sidor::wildfires"

# Safety gate for the last cell - real Anki write only happens if True
RUN_FOR_REAL = True

## 1. Tag the covering phrase(s) in Firestore

Finds the minimum set of existing phrases whose Swedish translation covers the vocab in `TEXT`, and tags them. This is a real Firestore write (low-risk/reversible - see `delete_tag_from_firestore` in `phrases/search.py` if you need to undo it).

In [ ]:
tagged_phrases, missing = add_tags_from_text(TEXT, LANGUAGE, TAG)

print(f"\nTagged {len(tagged_phrases)} phrase(s):")
for p in tagged_phrases:
    sv_text = p.translations[LANGUAGE].text
    print(f"  {p.key} | en: {p.english!r} | sv-SE: {sv_text!r}")
print(f"Missing vocab (no covering phrase found): {missing}")

In [ ]:
# generate missing phrases

In [ ]:
new_phrases = generate_phrases_from_vocab_dict(missing, TARGET_LANGUAGE, SOURCE_LANGUAGE, split_on_space=True, tags=[TAG])

In [ ]:
new_phrases = ['Jag berättar en historia',
 'Hon berättade om resan',
 'Ska du berätta sanningen?',
 'Vinden blåser starkt idag',
 'Hon blåste ut ljuset',
 'Jag ska blåsa ballongen',
 'Han blåste på flöjten',
 'Jag släcker lampan nu',
 'Hon släckte ljuset igår',
 'Ska du släcka ljusen?',
 'Släck elden med vatten',
 'Viruset sprider sig snabbt',
 'Hon spred smör på brödet',
 'Kommer du sprida nyheten?',
 'Sprid filten på marken',
 'De sprider rykten',
 'Jag stoppar bilen här',
 'Han stoppade mig igår',
 'Kommer du stoppa dem?',
 'Stoppa sockorna i lådan',
 'Kan du stoppa hålet?',
 'De tvingar oss att arbeta',
 'Han tvingade mig att stanna',
 'Kommer du tvinga honom?',
 'Jag tvingas lämna landet',
 'askan i kaminen',
 'kontrollen över situationen',
 'det sydvästra området',
 'personalen i skogen',
 'den mörka skogen',
 'den sydvästra delen']

In [ ]:
# create the phrases
ALL_NEW_PHRASES = []
for phrase in new_phrases:
    p = Phrase.create_from_foreign(phrase, TARGET_LANGUAGE, split_on_space=True, tags=[TAG])
    ALL_NEW_PHRASES.append(p)

In [ ]:
for phrase in ALL_NEW_PHRASES:
    sv_text = phrase.translations[TARGET_LANGUAGE.to_tag()].text
    print(f"{phrase.english} |  {sv_text}")

In [ ]:
for p in ALL_NEW_PHRASES:
    p.generate_audio(context="flashcard", language=TARGET_LANGUAGE,split_on_space=True)
    p.upload(language=TARGET_LANGUAGE)
    p.generate_image()
    p.upload()


## 2. Confirm the tag in Firestore, unprefixed

In [ ]:
P2 = find_phrases_by_tag(TAG, TARGET_LANGUAGE)

In [ ]:
[p.english for p in P2]

In [ ]:

FINAL_PHRASES = tagged_phrases + ALL_NEW_PHRASES
FINAL_PHRASES = sorted(FINAL_PHRASES, key=lambda x: len(x.translations[str(TARGET_LANGUAGE)].text))


In [ ]:

for p in FINAL_PHRASES:
    print(p.key, "->", p.translations[TARGET_LANGUAGE.to_tag()].tags)

In [ ]:
create_anki_deck(
        FINAL_PHRASES,
        source_language=SOURCE_LANGUAGE,
        target_language=TARGET_LANGUAGE,
        output_path=f"../outputs/decks/{TARGET_LANGUAGE.to_tag()}/{TARGET_LANGUAGE.to_tag()}-{TAG.replace('::', '-')}.apkg",
        deck_name=f"FirePhrase - {TARGET_LANGUAGE.language_name()}::{TAG}",
    )

## 3. Sync the tag into the live Anki collection (dry run)

`dry_run=True` makes zero writes anywhere - safe to re-run as many times as you like.

In [ ]:
col = get_anki_collection()
try:
    report = sync_tag_to_anki(col, TAG, SOURCE_LANGUAGE, LANGUAGE, dry_run=False)
    print(report.summary())
    for r in report.results:
        print(" ", r)
finally:
    close_anki_collection()

## 4. Run for real — only after reviewing the dry-run report above

Close Anki Desktop first (it holds an exclusive lock on the collection file). Set `RUN_FOR_REAL = True` in the config cell and re-run it, then run this cell. Takes a real Anki backup before writing, same as `scripts/sync_anki_tags.py`.

In [ ]:
import os
from pathlib import Path

if not RUN_FOR_REAL:
    print("Skipped - set RUN_FOR_REAL = True in the config cell above and re-run both cells when ready.")
else:
    col = get_anki_collection()
    try:
        backup_folder = str(Path(os.environ["ANKI_COLLECTION_PATH"]).parent / "backups")
        os.makedirs(backup_folder, exist_ok=True)
        col.create_backup(backup_folder=backup_folder, force=True, wait_for_completion=True)

        report = sync_tag_to_anki(col, TAG, SOURCE_LANGUAGE, LANGUAGE, dry_run=False)
        print(report.summary())
        for r in report.results:
            print(" ", r)
    finally:
        close_anki_collection()